# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the record sets along with their `@id` fields, and print their fields/columns and `@id`s for inspection.

In [ ]:
# Get all record sets
record_sets = dataset.metadata.recordSet

print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    # Print field IDs
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    {f['@id']} - {f.get('name','')}")
    # Print column IDs (if present)
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    {c['@id']} - {c.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll load all available record sets for analysis. Please refer to their `@id` shape in the previous overview.

In [ ]:
# Extract data from each record set
dataframes = {}

for rs in record_sets:
    record_set_id = rs['@id']
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields/Columns: {df.columns.tolist()}")
    print(df.head(2))
    print()
# Pick the first record set for demonstration below
main_record_set_id = record_sets[0]['@id'] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We'll use the first record set loaded above for demonstration. Adjust field IDs and column names as needed.

In [ ]:
# Example: Select a numeric field for analysis (e.g., GAD-7 score or PHQ-9)
# Field/column names may vary; inspect the DataFrame columns from above
df = dataframes[main_record_set_id]
print(f"Columns in main record set: {df.columns.tolist()}")

# Define field IDs by their names (replace as needed for your actual dataset columns)
# For demonstration, suppose 'gad7_score', 'phq9_score', 'psq_score', 'age', 'village', 'gender', etc. exist
numeric_field = 'gad7_score' if 'gad7_score' in df.columns else df.select_dtypes(include='number').columns[0]
print(f"Selected numeric field: {numeric_field}")

# Filtering example
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric column
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Example: Grouping by a field (e.g., gender or village)
group_field = 'gender' if 'gender' in df.columns else (df.select_dtypes(include='object').columns[0] if len(df.select_dtypes(include='object').columns) > 0 else None)
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll demonstrate a histogram and a boxplot for the selected numeric field, optionally grouped by a categorical field.

In [ ]:
# Histogram for the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Boxplot grouped by group_field
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich survey information regarding mental health indicators (e.g., GAD-7, PHQ-9, PSQ) for residents of Kilifi County, Kenya.
- Basic EDA reveals distribution characteristics and group differences in psychological scores.
- The Croissant schema facilitates clear structural understanding and reproducible access by referencing data entities via their `@id` fields.